# Student–Mentor Matching Algorithm

A weighted scoring algorithm that matches students to mentors based on stage alignment, confusion type, domain overlap, and mentoring style compatibility.

** Approach: Explicit weighted scoring rather than ML embeddings. Each signal maps directly to a schema field
---

## Mentor Dataset

In [6]:
import json
import json

with open("mentors.json", "r") as f:
    MENTORS = json.load(f)



---
## Student details
Change the student context here.

In [7]:
# ── STUDENT INPUT ──────────────────────────────────────────────────────────────
# Edit the fields below to change the student context.
# arbitary student data
# The matching algorithm reads directly from this dict.

STUDENT = {
    # ── Common fields ──
    "year": 2,                                  # 1 / 2 / 3 / 4
    "branch": "CSE",
    "personality": "Introvert",                 # Introvert / Extrovert / Ambivert
    "problem_category": "Career",               # Academic / Career / Personal / Time Management / FOMO
    "confusion_type": "Indecision",             # Indecision / Lack of Information / Overwhelmed / FOMO
    "interest_fields": ["Software", "Data Science"],
    "interest_certainty": "Low",                # Low / Medium / High
    "decision_urgency": "Immediate",            # Immediate / Exploring
    "guidance_preference": "Help me think it through",  # Tell me what to do / Help me think it through / Just share your experience

    # ── Year-2 specific ──
    "primary_focus": ["Internship"],            # list — Internship / Courses / Career Direction / Skill Development / Exploration
    "core_vs_noncore": "Undecided",

    # ── Internship sub-fields (active because primary_focus includes Internship) ──
    "internship_status": "Preparing",           # Not started / Preparing / Applied / Secured
    "internship_concern": ["Don't know how to prepare", "Not getting shortlisted"],
    "target_domains": ["Software"],
    "resume_preparedness": "Basic",             # None / Basic / Strong
}

print("Student context loaded:")
for k, v in STUDENT.items():
    print(f"  {k}: {v}")

Student context loaded:
  year: 2
  branch: CSE
  personality: Introvert
  problem_category: Career
  confusion_type: Indecision
  interest_fields: ['Software', 'Data Science']
  interest_certainty: Low
  decision_urgency: Immediate
  guidance_preference: Help me think it through
  primary_focus: ['Internship']
  core_vs_noncore: Undecided
  internship_status: Preparing
  internship_concern: ["Don't know how to prepare", 'Not getting shortlisted']
  target_domains: ['Software']
  resume_preparedness: Basic


---
## Scoring Engine
Weighted scoring across 6 signal groups. Each signal has an explicit weight and rationale.

In [ ]:
# ── SCORING ENGINE ─────────────────────────────────────────────────────────────
#
# Signal weights (sum to 100):
#
#  S1  Stage alignment         25   Most important — wrong stage = irrelevant advice
#  S2  Confusion type match    20   Core thesis of schema: match HOW student is confused
#  S3  Domain overlap          20   Ensures content relevance
#  S4  Advising confidence     15   Prevents bad sessions from unconfident mentors
#  S5  Mentoring style match   12   Communication compatibility reduces friction
#  S6  Recency bonus            8   Recent grads have more relevant context
#                             ---
#                             100
# in the more advanced implementation we can make these weights dynamic , by updating them 
import math
from datetime import datetime

CURRENT_YEAR = datetime.now().year
WEIGHTS = {"stage": 25, "confusion": 20, "domain": 20, "advising": 15, "style": 12, "recency": 8}

# Helpers ──────────────────────────────────────────────────────────────────────

CONFUSION_COMPAT = {
    # student confusion → mentor confusion types that help most
    "Indecision":         {"Indecision": 1.0, "FOMO": 0.7, "Overwhelmed": 0.5, "Lack of Information": 0.3},
    "Lack of Information": {"Lack of Information": 1.0, "Indecision": 0.5, "Overwhelmed": 0.3, "FOMO": 0.2},
    "Overwhelmed":        {"Overwhelmed": 1.0, "Indecision": 0.6, "Time Management": 0.8, "FOMO": 0.4},
    "FOMO":               {"FOMO": 1.0, "Indecision": 0.7, "Overwhelmed": 0.5, "Lack of Information": 0.3},
}

STYLE_COMPAT = {
    # (student_personality, guidance_preference) → best mentor styles
    ("Introvert",  "Help me think it through"):  {"Facilitative": 1.0, "Balanced": 0.7, "Directive": 0.2},
    ("Introvert",  "Tell me what to do"):         {"Directive": 1.0, "Balanced": 0.7, "Facilitative": 0.4},
    ("Introvert",  "Just share your experience"): {"Facilitative": 0.9, "Balanced": 1.0, "Directive": 0.5},
    ("Extrovert",  "Help me think it through"):  {"Balanced": 1.0, "Directive": 0.8, "Facilitative": 0.6},
    ("Extrovert",  "Tell me what to do"):         {"Directive": 1.0, "Balanced": 0.8, "Facilitative": 0.5},
    ("Extrovert",  "Just share your experience"): {"Directive": 0.8, "Balanced": 1.0, "Facilitative": 0.7},
    ("Ambivert",   "Help me think it through"):  {"Balanced": 1.0, "Facilitative": 0.8, "Directive": 0.6},
    ("Ambivert",   "Tell me what to do"):         {"Directive": 1.0, "Balanced": 0.9, "Facilitative": 0.6},
    ("Ambivert",   "Just share your experience"): {"Balanced": 1.0, "Facilitative": 0.8, "Directive": 0.7},
}

CONF_VAL = {"High": 1.0, "Medium": 0.6, "Low": 0.2}

def year_key(yr):
    return f"Y{yr}"

# ── Signal scorers ─────────────────────────────────────────────────────────────

def score_stage(student, mentor):
    
    # S1 — Stage alignment (25 pts)
    # Does the mentor have a year bucket matching the student's current year?
    # Also rewards mentors who navigated a similar decision in that year.

    yk = year_key(student["year"])
    journey = mentor.get("year_journey", {})

    if yk not in journey:
        return 0.0, "No matching year in journey"

    year_data = journey[yk]
    score = 0.7  # base: mentor has this year documented

    # Bonus: primary decision aligns with student's focus
    student_focus_keywords = " ".join(student.get("primary_focus", [])).lower()
    mentor_decision = year_data.get("primary_decision", "").lower()
    if any(kw in mentor_decision for kw in student_focus_keywords.split()):
        score += 0.3

    return min(score, 1.0), f"Year {yk} present; decision: '{year_data.get('primary_decision', 'N/A')}'"


def score_confusion(student, mentor):
    # S2 — Confusion type match (20 pts)
    # Matches the student's confusion type against confusion types the mentor
    # faced in the relevant year. Uses compatibility matrix, not exact match allows for mentors who had different but related struggles to still be helpful as confusions might be interrelated 
    student_conf = student.get("confusion_type", "")
    yk = year_key(student["year"])
    journey = mentor.get("year_journey", {})

    if yk not in journey:
        return 0.0, "Year missing"

    mentor_conf = journey[yk].get("confusion_type", "")
    compat = CONFUSION_COMPAT.get(student_conf, {})
    score = compat.get(mentor_conf, 0.1)

    return score, f"Student: {student_conf} | Mentor had: {mentor_conf} | Compat: {score:.2f}"


def score_domain(student, mentor):
    # S3 — Domain overlap (20 pts)
    # Checks if the mentor explored or worked in the student's interest domains.
    # Rewards deeper exploration stages (Worked > Achieved > Attempted > Researched).
    student_domains = set(d.lower() for d in student.get("interest_fields", []) + student.get("target_domains", []))
    yk = year_key(student["year"])
    journey = mentor.get("year_journey", {})

    stage_weights = {"Worked": 1.0, "Achieved": 0.85, "Attempted": 0.6, "Researched": 0.35}
    best = 0.0
    matched = []

    # Check year-specific domains
    if yk in journey:
        for d in journey[yk].get("domains_considered", []):
            if d.lower() in student_domains:
                # Exploration stage not always stored per-domain; default to Attempted
                stage = "Attempted"
                w = stage_weights.get(stage, 0.5)
                if w > best:
                    best = w
                matched.append(d)

    # Current domain match (mentor ended up where student wants to go)
    if mentor.get("current_domain", "").lower() in student_domains:
        best = max(best, stage_weights["Worked"])
        matched.append(f"{mentor['current_domain']} (current role)")

    detail = f"Matched domains: {matched}" if matched else "No domain overlap"
    return best, detail


def score_advising(student, mentor):
    # S4 — Advising confidence (15 pts)
    # Checks if the mentor has high confidence advising on the student's problem category.
    # Low-confidence mentors on the relevant topic are penalised.
    topic_map = {
        "Career": ["Internships", "Placements", "Career direction", "Domain switching"],
        "Academic": ["Skill development", "Career direction"],
        "Time Management": ["Time management"],
        "FOMO": ["Career direction", "Domain switching"],
        "Personal": ["Career direction"]
    }

    student_cat = student.get("problem_category", "Career")
    relevant_topics = topic_map.get(student_cat, [])

    # Also check primary focus topics
    focus_to_topic = {
        "Internship": "Internships",
        "Courses": "Skill development",
        "Career Direction": "Career direction",
        "Skill Development": "Skill development",
        "Exploration": "Career direction"
    }
    for f in student.get("primary_focus", []):
        t = focus_to_topic.get(f)
        if t and t not in relevant_topics:
            relevant_topics.append(t)

    conf = mentor.get("advising_confidence", {})
    advising_topics = mentor.get("advising_topics", [])

    best_score = 0.0
    best_topic = "None"
    for topic in relevant_topics:
        if topic in advising_topics:
            raw = CONF_VAL.get(conf.get(topic, "Low"), 0.2)
            if raw > best_score:
                best_score = raw
                best_topic = topic

    return best_score, f"Best advising topic: '{best_topic}' (conf={conf.get(best_topic, 'N/A')})"


def score_style(student, mentor):
    # S5 — Mentoring style match (12 pts)
    # Maps (student personality, guidance preference) → mentor style.
    key = (student.get("personality", "Ambivert"), student.get("guidance_preference", "Help me think it through"))
    compat = STYLE_COMPAT.get(key, {"Balanced": 0.8, "Facilitative": 0.7, "Directive": 0.6})
    mentor_style = mentor.get("mentoring_style", "Balanced")
    score = compat.get(mentor_style, 0.5)
    return score, f"Mentor style: {mentor_style} | Student prefers: {key[1]} ({key[0]})"


def score_recency(student, mentor):
    # S6 — Recency (8 pts)
    # Recent grads have more current awareness of market, tools, competition.
    # Decays over ~5 years. Mentors within 2 years of graduation get full score.
    years_since_grad = CURRENT_YEAR - mentor.get("graduation_year", 2020)
    if years_since_grad <= 2:
        score = 1.0
    elif years_since_grad <= 4:
        score = 0.75
    elif years_since_grad <= 6:
        score = 0.5
    else:
        score = 0.25
    return score, f"Graduated {mentor.get('graduation_year', '?')} ({years_since_grad} yrs ago)"


# ── Master scoring function ────────────────────────────────────────────────────

def score_mentor(student, mentor):
    # Computes weighted total score for a mentor given a student context.
    # Returns (total_score, breakdown_dict).
    s1, d1 = score_stage(student, mentor)
    s2, d2 = score_confusion(student, mentor)
    s3, d3 = score_domain(student, mentor)
    s4, d4 = score_advising(student, mentor)
    s5, d5 = score_style(student, mentor)
    s6, d6 = score_recency(student, mentor)

    w = WEIGHTS
    total = (
        s1 * w["stage"] +
        s2 * w["confusion"] +
        s3 * w["domain"] +
        s4 * w["advising"] +
        s5 * w["style"] +
        s6 * w["recency"]
    )

    breakdown = {
        "Stage alignment":      (round(s1 * w["stage"], 2), d1),
        "Confusion type":       (round(s2 * w["confusion"], 2), d2),
        "Domain overlap":       (round(s3 * w["domain"], 2), d3),
        "Advising confidence":  (round(s4 * w["advising"], 2), d4),
        "Mentoring style":      (round(s5 * w["style"], 2), d5),
        "Recency":              (round(s6 * w["recency"], 2), d6),
    }

    return round(total, 2), breakdown


print("Scoring engine ready.")
print(f"\nSignal weights: {WEIGHTS}")
print(f"Total weight: {sum(WEIGHTS.values())} (must be 100)")

Scoring engine ready.

Signal weights: {'stage': 25, 'confusion': 20, 'domain': 20, 'advising': 15, 'style': 12, 'recency': 8}
Total weight: 100 (must be 100)


---
## Run Matching & Display Results

In [9]:
# ── RUN MATCHING ───────────────────────────────────────────────────────────────

results = []
for mentor in MENTORS:
    total, breakdown = score_mentor(STUDENT, mentor)
    results.append((total, mentor, breakdown))

results.sort(key=lambda x: x[0], reverse=True)

# ── Display ────────────────────────────────────────────────────────────────────

print("  STUDENT - MENTOR MATCH RESULTS")
print(f"  Student: Y{STUDENT['year']} | {STUDENT['branch']} | {STUDENT['problem_category']} | "
    f"Confusion: {STUDENT['confusion_type']}")
print(f"  Interests: {STUDENT['interest_fields']} | Personality: {STUDENT['personality']}")
print(f"  Guidance preference: {STUDENT['guidance_preference']}")

for rank, (score, mentor, breakdown) in enumerate(results, 1):

    print(f"\n#{rank:>2}  {mentor['name']:<22}   {score:5.1f}/100")
    print(f"     {mentor['current_role']}  |  {mentor['branch']}  |  Style: {mentor['mentoring_style']}  |  Grad: {mentor['graduation_year']}")

    # Signal breakdown
    for signal, (pts, detail) in breakdown.items():
        print(f"     {'·':>3} {signal:<22} {pts:>5.1f}pt  —  {detail}")


    if rank > 5:
        print(f"#{rank:>2}  {mentor['name']:<22}  {score:5.1f}/100  |  {mentor['current_role']}")

  STUDENT - MENTOR MATCH RESULTS
  Student: Y2 | CSE | Career | Confusion: Indecision
  Interests: ['Software', 'Data Science'] | Personality: Introvert
  Guidance preference: Help me think it through

# 1  Arjun Sharma              98.0/100
     SDE at Microsoft  |  CSE  |  Style: Facilitative  |  Grad: 2023
       · Stage alignment         25.0pt  —  Year Y2 present; decision: 'Internship vs exploration'
       · Confusion type          20.0pt  —  Student: Indecision | Mentor had: Indecision | Compat: 1.00
       · Domain overlap          20.0pt  —  Matched domains: ['Software', 'Software (current role)']
       · Advising confidence     15.0pt  —  Best advising topic: 'Internships' (conf=High)
       · Mentoring style         12.0pt  —  Mentor style: Facilitative | Student prefers: Help me think it through (Introvert)
       · Recency                  6.0pt  —  Graduated 2023 (3 yrs ago)

# 2  Tanvi Shah                90.4/100
     SDE at Atlassian  |  CSE  |  Style: Directive  |  